# Two Pointers & Sliding Window

KTH  
July 12, 2026

---
**Ref** **Part I** from EPI **Chapter 14 — Greedy Algorithms and Invariants**: the invariants derives two pointers from first principles. **Part II** (sliding window) is *supplementary
material written in the same style, not from EPI*, since the book has no
coverage of it.

# Part I — EPI Chapter 14: Greedy Algorithms and Invariants

*(Extracted from EPI. The "Invariants" half is the formal foundation of the
two-pointer technique.)*

## 1. Greedy algorithms

Sometimes there are **multiple greedy algorithms** for a problem, and only some
are optimum.

**Example — pairing cities.** Consider `2n` cities on a line, half white, half
black. Pair white with black cities one-to-one so the total length of road
needed to connect paired cities is minimized (pairs may share road sections).

- **Naive greedy (suboptimal):** scan the white cities and pair each with the
  closest unpaired black city. With white cities at 0 and 3 and black cities at
  2 and 5: processing the white city at 3 first pairs it with 2, forcing 0 with
  5 — road length 5. But pairing (0, 2) and (3, 5) gives length 4.
- **Correct greedy:** iterate through **all** cities left-to-right, pairing each
  city with the nearest unpaired city of *opposite color*. The first city's
  pairing must be optimum (repairing it with its nearest opposite-color city
  never adds road), which seeds an inductive proof of overall optimality.

**Lesson:** the correct greedy algorithm is not always the obvious one.

## 2. Greedy algorithms — making change

For US currency (coins of 1, 5, 10, 25, 50, 100 cents), the greedy algorithm
for making change yields the minimum number of coins. The hallmark of a greedy
algorithm: once it selects the number of coins of a particular value, it
**never revisits** that selection.

> ⚠️ The book's code uses `cents / coin`, which in Python 3 is *float*
> division — a Python 2 leftover. Below it's corrected to floor division `//`.

In [1]:
def change_making(cents):
    COINS = [100, 50, 25, 10, 5, 1]
    num_coins = 0
    for coin in COINS:
        num_coins += cents // coin   # book has `/` (Python 2); `//` is correct in Python 3
        cents %= coin
    return num_coins

print(change_making(87))   # 50 + 25 + 10 + 1 + 1 = 5 coins
print(change_making(6))    # 5 + 1 = 2 coins

5
2


We perform 6 iterations with constant work each, so the time complexity is
`O(1)`.

### Top Tips for Greedy Algorithms
- A greedy algorithm is often the right choice for an **optimization problem
  with a natural set of choices** to select from.
- It's often easier to **conceptualize greedily recursively**, then implement
  iteratively for performance.
- Even when greedy doesn't yield the optimum, it can give **insight into the
  optimum algorithm**, or serve as a heuristic.
- Sometimes the correct greedy algorithm is **not obvious**.

## 3. Invariants

An **invariant** is a condition that holds true throughout the execution of a
program — on variable values or on control logic. A well-chosen invariant rules
out potential solutions that are suboptimal or dominated by others.

Classic examples:
- **Binary search** maintains the invariant that the candidate space contains
  all possible solutions as it executes.
- **Selection sort** works with successively larger subarrays starting at index
  0, preserving the invariant that the subarray is sorted, its elements are ≤
  the remaining elements, and the whole array stays a permutation of the
  original.

## 4. Invariants — two-sum in a sorted array (*the two-pointer technique*)

**Problem.** Given a **sorted** array and a target, determine whether two
entries sum to the target. For ⟨−2, 1, 2, 4, 7, 11⟩: entries sum to 6 and to
10, but not to 0 or 13.

Three approaches, in derivation order (brute force → identify waste → reframe):

| Approach | Time | Space |
|---|---|---|
| Nested loops over all pairs | `O(n²)` | `O(1)` |
| Hash table: for each `e`, test `K − e` | `O(n)` | `O(n)` |
| **Invariant / two pointers** | `O(n)` | `O(1)` |

**The invariant.** Maintain a subarray `A[i..j]` that is **guaranteed to hold a
solution, if one exists**. Initialize it to the whole array and iteratively
shrink from one side or the other, exploiting sortedness:
- If `A[i] + A[j] < target`, then `A[i]` can never combine with *any* element
  to reach the target (`A[j]` is the largest available) — discard the left end.
- Symmetrically, if the sum is too large, discard the right end.

In [2]:
def has_two_sum(A, t):
    i, j = 0, len(A) - 1
    while i <= j:
        if A[i] + A[j] == t:
            return True
        elif A[i] + A[j] < t:
            i += 1          # A[i] can't be part of any solution — shrink left
        else:               # A[i] + A[j] > t.
            j -= 1          # A[j] can't be part of any solution — shrink right
    return False

A = [-2, 1, 2, 4, 7, 11]
for target in (6, 10, 0, 13):
    print(f"target {target:>2}: {has_two_sum(A, target)}")

target  6: True
target 10: False
target  0: True
target 13: True


**Complexity.** `O(n)` time, `O(1)` space — the subarray is represented by
just two index variables. *(This is exactly LeetCode 167, Two Sum II, already in
the solutions repo — and the inner engine of 15, 3Sum.)*

### Top Tips for Invariants
- Identifying the right invariant is an art. The key strategy: **work small
  examples to hypothesize the invariant.**
- Often the invariant is a **subset of the input space** — e.g., a subarray.

# Part II — Sliding Window *(supplementary — not from EPI)*

## 5. The sliding window idea

A sliding window is a **two-pointer pattern specialized to contiguous
subarrays/substrings**: both pointers move in the *same* direction (left to
right), delimiting a window `[left, right]`, and the algorithm maintains an
invariant about the window's contents.

**When it applies.** The problem asks about contiguous subarrays/substrings
(longest, shortest, count, existence) *and* window validity is **monotone**:
- growing an invalid window keeps it invalid (or growing a valid window keeps
  it valid, depending on direction).

That monotonicity is what lets each pointer advance without ever backtracking —
giving `O(n)` total, since each pointer moves at most `n` steps.

**Derivation (brute force → waste → reframe).** Brute force checks all
`O(n²)` windows, re-scanning each in `O(n)` — `O(n³)`. The waste: adjacent
windows share almost all their content. Reframe: maintain a running summary
(count map, sum, distinct count) that's **updated incrementally** — add the
entering element, remove the leaving element — instead of recomputed.

## 6. Fixed-size window

The window length `k` is given. Slide right one step at a time; update the
summary in `O(1)` per step.

**Example — maximum sum of any subarray of length `k`.**

In [3]:
def max_sum_fixed_window(A, k):
    if k > len(A):
        return None
    window_sum = sum(A[:k])            # first window, computed once
    best = window_sum
    for right in range(k, len(A)):
        window_sum += A[right] - A[right - k]   # enter right, leave left
        best = max(best, window_sum)
    return best

print(max_sum_fixed_window([2, 1, 5, 1, 3, 2], 3))   # 9  (5+1+3)

9


*(LeetCode 567, Permutation in String — already in the repo — is the same
skeleton with a character-count map as the summary and "counts match" as the
window test.)*

## 7. Variable-size window (grow/shrink)

The dominant form. `right` expands the window each iteration; a `while` loop
advances `left` to restore the invariant whenever it breaks. The answer is read
off each time the window is valid.

**Template:**

```text
for right in range(n):
    add A[right] to summary
    while window invalid:
        remove A[left] from summary
        left += 1
    record answer from window [left, right]
```

**Example — longest substring without repeating characters** *(LeetCode 3,
already in the repo)*. Invariant: the window contains no duplicate characters.

In [4]:
def longest_no_repeat(s):
    seen = set()          # summary: characters currently in the window
    left = 0
    best = 0
    for right, c in enumerate(s):
        while c in seen:              # invariant broken: c would be a duplicate
            seen.remove(s[left])      # shrink from the left until restored
            left += 1
        seen.add(c)
        best = max(best, right - left + 1)
    return best

print(longest_no_repeat("abcabcbb"))   # 3 ("abc")
print(longest_no_repeat("bbbbb"))      # 1
print(longest_no_repeat("pwwkew"))     # 3 ("wke")

3
1
3


**Why this is `O(n)`:** `right` advances `n` times; `left` only ever
advances, at most `n` times total across the whole run — so the inner `while`
is amortized `O(1)` per step. Space is `O(min(n, |alphabet|))` for the summary.

**Example — minimum-size subarray with sum ≥ target.** Same template, opposite
optimization direction: shrink *while valid* to find the smallest window.

In [5]:
def min_subarray_len(target, A):
    left = 0
    window_sum = 0
    best = float('inf')
    for right, x in enumerate(A):
        window_sum += x
        while window_sum >= target:            # valid: try to shrink
            best = min(best, right - left + 1)
            window_sum -= A[left]
            left += 1
    return 0 if best == float('inf') else best

print(min_subarray_len(7, [2, 3, 1, 2, 4, 3]))   # 2  ([4, 3])

2


## 8. Top tips for sliding window

- **Recognize the shape:** "longest/shortest/count of contiguous
  subarray/substring satisfying a condition" is the signature.
- **Check monotonicity first.** If extending a window can flip it between
  valid and invalid *in both directions* (e.g., constraints on an exact sum
  with negative numbers allowed), a plain sliding window doesn't apply — reach
  for prefix sums + hash table instead.
- **Choose the summary structure** so both *add* and *remove* are `O(1)`:
  running sum, `Counter`, distinct-count, max-frequency (as in LeetCode 424,
  Longest Repeating Character Replacement, already in the repo).
- **Longest-window problems** shrink while *invalid*; **shortest-window
  problems** shrink while *valid*. The template is the same; only the shrink
  condition and where you record the answer differ.
- **Fixed-`k` problems** need no `while` loop — enter one element, evict one
  element, every step.
- Relationship to Part I: converging two pointers (two-sum) exploit
  **sortedness**; sliding windows exploit **contiguity + monotone validity**.
  Both are invariant-maintenance algorithms with `O(n)` pointer budgets.